In [1]:
import os
import zipfile
import pickle
import pandas as pd
from tqdm import tqdm  # Import tqdm for the progress bar

def load_dataframe_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Iterate through all files in the zip
        for file_name in zip_ref.namelist():
            if file_name.endswith('.pkl'):
                with zip_ref.open(file_name) as file:
                    # Load the data from the pickle file
                    data = pickle.load(file)
                    
                    # Check if the data is a list, and if so, convert it to a DataFrame
                    if isinstance(data, list):
                        df = pd.DataFrame(data, columns=['cik', 'date', 'note_text'])
                        df['item_7'] = df['note_text'].apply(extract_item_7)
                        df['risk_factors'] = df['note_text'].apply(extract_risk_factors)
                        # print(df[['cik', 'date','item_7','risk_factors']].values[0])
                        # print(df[['cik', 'date']].values[0])
                        # print(df['item_7'].values[0])
                        # print(df['risk_factors'].values[0])
                        
                        return df[['cik', 'date','item_7','risk_factors']]
                    else:
                        print(f"Warning: The file {file_name} does not contain a list or DataFrame.")
    return None

def combine_pickles_in_folders(folders):
    combined_df = pd.DataFrame()
    
    # Loop through each folder
    for folder in folders:
        for root, dirs, files in os.walk(folder):
            # Add tqdm to show progress when iterating over the files
            for file in tqdm(files, desc=f"Processing files in {folder}", unit="file"):
                if file.endswith('.zip'):
                    zip_path = os.path.join(root, file)
                    # print(f"Processing {zip_path}")
                    
                    # Load the dataframe from the zip file
                    df = load_dataframe_from_zip(zip_path)
                    if df is not None:
                        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df




In [2]:
def extract_risk_factors(text):
    # Normalize the text to handle different cases and line breaks
    normalized_text = text.lower()

    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))
    
    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())
    
    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None

In [3]:
import re

def extract_risk_factors(text):
    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r"(?:\nITEM\s*1A|\nItem\s*1A)(.*?)(?=\nITEM\s*\d+|\nItem\s*\d+)"

    # Find all matches
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        # Concatenate all the matches into a single string
        concatenated_text = "\n\n".join(match.strip() for match in matches)
        return concatenated_text
    return None


def extract_item_7(text):
    # Use a case-insensitive search for 'ITEM 7' or 'Item 7'
    pattern = r"(?:\nITEM\s*7|\nItem\s*7)(.*?)(?=\nITEM\s*\d+|\nItem\s*\d+)"

    # Find all matches
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        # Concatenate all the matches into a single string
        concatenated_text = "\n\n".join(match.strip() for match in matches)
        return concatenated_text
    return None


In [ ]:


# STEP 3: Define helper functions
def extract_cik(filename):
    match = re.search(r'_edgar_data_(\d+)_', filename)

    if match:
        cik = match.group(1)
        return cik
    else:
        print("CIK not found:",filename)

def extract_item_7(text):
    normalized_text = text.lower()

    # New pattern: anchors to line beginnings, and uses \b for better word boundary matching
    pattern = r'^\s*item\s+7\.?\s*(.*?)(?=^\s*item\s+7a\.?|^\s*item\s+8\b|^\s*item\s+\d+\s*\b|$\Z)'

    match = re.search(pattern, normalized_text, re.IGNORECASE | re.DOTALL | re.MULTILINE)

    if match:
        # Use the start and end index from the original text
        content_start, content_end = match.start(1), match.end(1)
        return [text[content_start:content_end].strip()]

    return None

def extract_item_7(text):
    normalized_text = text.lower()

    # New pattern: anchors to line beginnings, and uses \b for better word boundary matching
    pattern = r'[\r\n]+\s*item[\s\n]*7[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*7[\s\n]*a[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*8[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))

    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())

    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*7[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None

def extract_risk_factors(text):
    # Normalize the text to handle different cases and line breaks
    normalized_text = text.lower()

    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))

    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())

    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None




In [5]:
# Define the folders to search
folders = ['../processed_data/processed_data_new','../processed_data/processed_data']

# Combine the dataframes
combined_dataframe = combine_pickles_in_folders(folders)

# Save the combined dataframe as a Parquet file
if not combined_dataframe.empty:
    combined_dataframe.to_parquet('combined_dataframe.parquet', compression='gzip')
    print("DataFrames combined and saved as 'combined_dataframe.parquet'")
else:
    print("No data found to combine.")

Processing files in ../processed_data/processed_data_new: 100%|██████████████████████| 41/41 [03:43<00:00,  5.46s/file]
Processing files in ../processed_data/processed_data: 100%|████████████████████████| 266/266 [18:15<00:00,  4.12s/file]


DataFrames combined and saved as 'combined_dataframe.parquet'


In [6]:
import os
import zipfile
import pickle
import pandas as pd
from tqdm import tqdm  # Import tqdm for the progress bar

def load_dataframe_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Iterate through all files in the zip
        for file_name in zip_ref.namelist():
            if file_name.endswith('.pkl'):
                with zip_ref.open(file_name) as file:
                    # Load the data from the pickle file
                    data = pickle.load(file)
                    
                    # Check if the data is a list, and if so, convert it to a DataFrame
                    if isinstance(data, list):
                        df = pd.DataFrame(data, columns=['cik', 'date', 'note_text'])
                        df['item_7'] = df['note_text'].apply(extract_item_7)
                        df['risk_factors'] = df['note_text'].apply(extract_risk_factors)
                        return df[['cik', 'date','item_7','risk_factors']]
                    else:
                        print(f"Warning: The file {file_name} does not contain a list or DataFrame.")
    return None

def combine_pickles_in_folders(folders):
    combined_df = pd.DataFrame()
    
    # Loop through each folder
    for folder in folders:
        for root, dirs, files in os.walk(folder):
            # Add tqdm to show progress when iterating over the files
            for file in tqdm(files, desc=f"Processing files in {folder}", unit="file"):
                if file.endswith('.zip'):
                    zip_path = os.path.join(root, file)
                    # Load the dataframe from the zip file
                    df = load_dataframe_from_zip(zip_path)
                    if df is not None:
                        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df

def save_as_multiple_parquet(df, output_folder, num_files=10):
    # Split the DataFrame into `num_files` chunks
    chunk_size = len(df) // num_files
    for i in range(num_files):
        # Determine start and end index for each chunk
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size if i < num_files - 1 else len(df)
        
        # Create a chunk DataFrame
        chunk_df = df.iloc[start_idx:end_idx]
        
        # Define the file name
        output_path = os.path.join(output_folder, f"combined_part_{i+1}.parquet")
        
        # Save the chunk as a Parquet file
        chunk_df.to_parquet(output_path, compression='gzip', index=False)
        print(f"Saved chunk {i+1} as {output_path}")

# Define the folders to search
folders = ['../processed_data/processed_data_new', '../processed_data/processed_data']

# Combine the dataframes
combined_dataframe = combine_pickles_in_folders(folders)

# Save the combined dataframe as multiple Parquet files
if not combined_dataframe.empty:
    output_folder = 'output'  # Replace with your desired output folder
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    save_as_multiple_parquet(combined_dataframe, output_folder, num_files=10)
else:
    print("No data found to combine.")


Processing files in ../processed_data/processed_data_new: 100%|██████████████████████| 41/41 [03:55<00:00,  5.75s/file]
Processing files in ../processed_data/processed_data: 100%|████████████████████████| 266/266 [18:24<00:00,  4.15s/file]


Saved chunk 1 as output\combined_part_1.parquet
Saved chunk 2 as output\combined_part_2.parquet
Saved chunk 3 as output\combined_part_3.parquet
Saved chunk 4 as output\combined_part_4.parquet
Saved chunk 5 as output\combined_part_5.parquet
Saved chunk 6 as output\combined_part_6.parquet
Saved chunk 7 as output\combined_part_7.parquet
Saved chunk 8 as output\combined_part_8.parquet
Saved chunk 9 as output\combined_part_9.parquet
Saved chunk 10 as output\combined_part_10.parquet
